In [ ]:
# 使用纽约市出租车和豪华轿车委员会（NYC TLC）公开的真实订单数据。该网站按月提供Parquet格式数据，包含上下车时间、区域、里程、车费、支付方式等字段，单月即可达到数百万条记录。NYC TLC官方数据
# 项目要解决的问题
# 不同日期和时段的订单量如何变化？
# 哪些区域属于高需求、高收入区域？
# 哪些区域存在空驶或供需不匹配？
# 订单里程、时长、客单价和小费率有什么关系？
# 工作日、周末和高峰期的经营表现有何差异？
# 如何制定分时段、分区域的运力调度方案？

In [ ]:
#导包
import pandas as pd

In [ ]:
#数据的读取
jan = pd.read_parquet("D:\DEVELOP\Python-Project\PandasProject02\yellow_tripdata_2024-01.parquet")
feb = pd.read_parquet("D:\DEVELOP\Python-Project\PandasProject02\yellow_tripdata_2024-02.parquet")
mar = pd.read_parquet("D:\DEVELOP\Python-Project\PandasProject02\yellow_tripdata_2024-03.parquet")
#数据的合并
taxi_df = pd.concat([jan, feb, mar], ignore_index=True)

# 1 原始数据的筛查

In [ ]:
# 输出数据集行数和列数
print(f"数据维度：{taxi_df.shape}")
print(f"订单总数：{taxi_df.shape[0]:,}")
print(f"字段总数：{taxi_df.shape[1]}")

In [ ]:
# 检查字段名称和数据类型
for number, column in enumerate(taxi_df.columns, start=1):
    print(f"{number:>2}. {column}")
print(taxi_df.info())

In [ ]:
# 查看前5条数据

# 显示全部字段，避免Pandas省略中间列
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(taxi_df.head())

In [ ]:
# 检查订单时间范围
print(
    "最早上车时间：",
    taxi_df["tpep_pickup_datetime"].min()
)

print(
    "最晚上车时间：",
    taxi_df["tpep_pickup_datetime"].max()
)

print(
    "最早下车时间：",
    taxi_df["tpep_dropoff_datetime"].min()
)

print(
    "最晚下车时间：",
    taxi_df["tpep_dropoff_datetime"].max()
)

In [ ]:
# 检查缺失值
missing_summary = pd.DataFrame({
    # 每个字段的缺失数量
    "missing_count": taxi_df.isna().sum(),

    # 每个字段的缺失比例
    "missing_rate": taxi_df.isna().mean()
})

# 只展示存在缺失值的字段
missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].copy()

# 将缺失比例转换为百分比
missing_summary["missing_rate"] = (
    missing_summary["missing_rate"] * 100
).round(2)

# 按缺失数量从高到低排列
missing_summary = missing_summary.sort_values(
    by="missing_count",
    ascending=False
)

print(missing_summary)

In [ ]:
# 检查完全重复的订单
duplicate_count = taxi_df.duplicated().sum()
duplicate_rate = duplicate_count / len(taxi_df) * 100

print(f"完全重复记录数：{duplicate_count:,}")
print(f"完全重复记录比例：{duplicate_rate:.4f}%")

In [ ]:
# 检查核心数值字段
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

print(
    taxi_df[numeric_columns]
    .describe()
    .round(2)
    .T
)

In [ ]:
# 检查主要分类字段
print(
    taxi_df["payment_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n乘客数量分布：")

print(
    taxi_df["passenger_count"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# 检查明显异常值数量
# 上车时间晚于或等于下车时间
invalid_time_count = (
    taxi_df["tpep_pickup_datetime"]
    >= taxi_df["tpep_dropoff_datetime"]
).sum()

# 行程距离小于或等于0
invalid_distance_count = (
    taxi_df["trip_distance"] <= 0
).sum()

# 总金额小于或等于0
invalid_amount_count = (
    taxi_df["total_amount"] <= 0
).sum()

# 乘客数量小于或等于0
invalid_passenger_count = (
    taxi_df["passenger_count"] <= 0
).sum()

print(f"时间异常订单数：{invalid_time_count:,}")
print(f"里程小于等于0的订单数：{invalid_distance_count:,}")
print(f"总金额小于等于0的订单数：{invalid_amount_count:,}")
print(f"乘客数小于等于0的订单数：{invalid_passenger_count:,}")

# 2.原始订单数据清洗

In [ ]:
# 清洗原则：
# 1. 保留正常完成的2024年第一季度订单；
# 2. 删除时间、里程、金额和区域明显异常的订单；
# 3. 不因乘客数缺失而删除整条订单；
# 4. 记录每项规则删除的数据量，保证清洗过程可以解释。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# 使用copy()创建独立副本
clean_df = taxi_df.copy()

original_count = len(clean_df)

print("开始清洗纽约出租车订单数据")

print(f"原始订单数：{original_count:,}")


In [ ]:
1# 删除完全重复记录

before_count = len(clean_df)

clean_df = clean_df.drop_duplicates()

after_count = len(clean_df)

print(
    f"删除完全重复记录："
    f"{before_count - after_count:,} 条"
)


In [ ]:
# 统一时间字段的数据类型

# errors="coerce"表示：
# 如果某些值无法转换为时间，则转换为缺失值NaT
clean_df["tpep_pickup_datetime"] = pd.to_datetime(
    clean_df["tpep_pickup_datetime"],
    errors="coerce"
)

clean_df["tpep_dropoff_datetime"] = pd.to_datetime(
    clean_df["tpep_dropoff_datetime"],
    errors="coerce"
)


In [ ]:
# 限制研究时间为2024年第一季度


before_count = len(clean_df)

# 订单所属月份以上车时间为准
date_mask = (
    clean_df["tpep_pickup_datetime"].ge("2024-01-01")
    & clean_df["tpep_pickup_datetime"].lt("2024-04-01")
)

clean_df = clean_df.loc[date_mask].copy()

after_count = len(clean_df)

print(
    f"删除非2024年第一季度订单："
    f"{before_count - after_count:,} 条"
)


In [ ]:
# 计算订单行程时长
# ============================================================

# 时间差默认单位为秒
clean_df["trip_duration_minutes"] = (
    clean_df["tpep_dropoff_datetime"]
    - clean_df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print("\n清洗前行程时长统计：")

print(
    clean_df["trip_duration_minutes"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .round(2)
)

In [ ]:
# 删除行程时间异常订单


before_count = len(clean_df)

# 保留1～180分钟的订单
duration_mask = clean_df["trip_duration_minutes"].between(
    1,
    180,
    inclusive="both"
)

clean_df = clean_df.loc[duration_mask].copy()

after_count = len(clean_df)

print(
    f"删除行程时间异常订单："
    f"{before_count - after_count:,} 条"
)



In [ ]:
# 删除里程异常订单


before_count = len(clean_df)

# 保留大于0且不超过100英里的行程
distance_mask = clean_df["trip_distance"].between(
    0.01,
    100,
    inclusive="right"
)

clean_df = clean_df.loc[distance_mask].copy()

after_count = len(clean_df)

print(
    f"删除行程里程异常订单："
    f"{before_count - after_count:,} 条"
)

In [ ]:
# 计算平均行驶速度

# 平均速度 = 行程里程 ÷ 行程小时数
clean_df["average_speed_mph"] = (
    clean_df["trip_distance"]12
    / (clean_df["trip_duration_minutes"] / 60)
)

print("\n平均速度清洗前统计：")

print(
    clean_df["average_speed_mph"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .round(2)
)


In [ ]:
# 删除平均速度明显异常订单

before_count = len(clean_df)

# 平均速度必须大于0且不超过80英里/小时
speed_mask = clean_df["average_speed_mph"].between(
    0,
    80,
    inclusive="right"
)

clean_df = clean_df.loc[speed_mask].copy()

after_count = len(clean_df)

print(
    f"删除平均速度异常订单："
    f"{before_count - after_count:,} 条"
)

In [ ]:

# 删除上下车区域编号异常订单


before_count = len(clean_df)

# TLC出租车区域编号通常为1～263
location_mask = (
    clean_df["PULocationID"].between(
        1,
        263,
        inclusive="both"
    )
    & clean_df["DOLocationID"].between(
        1,
        263,
        inclusive="both"
    )
)

clean_df = clean_df.loc[location_mask].copy()

after_count = len(clean_df)

print(
    f"删除区域编号异常订单："
    f"{before_count - after_count:,} 条"
)


In [ ]:
# 处理乘客数量

# 乘客数量缺失、等于0或明显异常时，
# 不删除整条订单，只将该字段改为空值。
#
# 这样订单仍可用于订单量、收入、时段和区域分析，
# 但不会进入人均指标或乘客数量相关分析。

invalid_passenger_mask = (
    clean_df["passenger_count"].isna()
    | clean_df["passenger_count"].le(0)
    | clean_df["passenger_count"].gt(6)
)

clean_df.loc[
    invalid_passenger_mask,
    "passenger_count"
] = np.nan

print(
    "乘客数量被标记为缺失的订单："
    f"{clean_df['passenger_count'].isna().sum():,} 条"
)


In [ ]:

# 清洗金额异常订单

before_count = len(clean_df)

# 正常完成订单需要满足：
# 1. 基础车费大于0
# 2. 总金额大于0
#
# 使用notna()排除可能存在的金额缺失值
amount_mask = (
    clean_df["fare_amount"].notna()
    & clean_df["total_amount"].notna()
    & clean_df["fare_amount"].gt(0)
    & clean_df["total_amount"].gt(0)
)

clean_df = clean_df.loc[amount_mask].copy()

after_count = len(clean_df)

print("金额异常订单清洗")

print(f"清洗前订单数：{before_count:,}")
print(f"删除金额异常订单：{before_count - after_count:,}")
print(f"清洗后订单数：{after_count:,}")

In [ ]:
# 处理部分分类字段缺失


# store_and_fwd_flag是类别字段。
# 缺失值标记为Unknown，便于后续统计。
clean_df["store_and_fwd_flag"] = (
    clean_df["store_and_fwd_flag"]
    .fillna("Unknown")
)

# RatecodeID缺失不代表订单无效。
# 保留为缺失值，暂时不填充为0，避免人为创造类别。


In [ ]:
# 增加经营分析所需的时间字段

# 日期
clean_df["pickup_date"] = (
    clean_df["tpep_pickup_datetime"].dt.date
)

# 月份
clean_df["pickup_month"] = (
    clean_df["tpep_pickup_datetime"].dt.month
)

# 小时
clean_df["pickup_hour"] = (
    clean_df["tpep_pickup_datetime"].dt.hour
)

# 星期编号：星期一为0，星期日为6
clean_df["weekday_number"] = (
    clean_df["tpep_pickup_datetime"].dt.dayofweek
)
# 英文星期名称
clean_df["weekday_name"] = (
    clean_df["tpep_pickup_datetime"].dt.day_name()
)

# 是否为周末
clean_df["is_weekend"] = (
    clean_df["weekday_number"].isin([5, 6])
)

# 将小时划分为经营分析常用时间段
clean_df["time_period"] = pd.cut(
    clean_df["pickup_hour"],
    bins=[-1, 5, 9, 16, 19, 23],
    labels=[
        "凌晨",
        "早高峰",
        "日间",
        "晚高峰",
        "夜间",
    ]
)

In [ ]:
# 增加经营效率指标


# 单位里程收入
clean_df["revenue_per_mile"] = (
    clean_df["total_amount"]
    / clean_df["trip_distance"]
)

# 单位时间收入：美元/小时
clean_df["revenue_per_hour"] = (
    clean_df["total_amount"]
    / (clean_df["trip_duration_minutes"] / 60)
)

# # 小费率
# # 使用基础车费作为分母，比使用总金额更容易解释
# clean_df["tip_rate"] = (
#     clean_df["tip_amount"]
#     / clean_df["fare_amount"]
# )


# 修正：计算小费率

# 先创建小费率字段，默认设置为缺失值
clean_df["tip_rate"] = np.nan

# 仅针对以下订单计算小费率：
# 1. payment_type为1，即信用卡支付；
# 2. tip_amount不缺失；
# 3. tip_amount大于等于0；
# 4. fare_amount大于0。
valid_tip_mask = (
    clean_df["payment_type"].eq(1)
    & clean_df["tip_amount"].notna()
    & clean_df["tip_amount"].ge(0)
    & clean_df["fare_amount"].gt(0)
)

clean_df.loc[valid_tip_mask, "tip_rate"] = (
    clean_df.loc[valid_tip_mask, "tip_amount"]
    / clean_df.loc[valid_tip_mask, "fare_amount"]
)

# 将正负无穷值转换为缺失值，防止影响统计计算
clean_df["tip_rate"] = clean_df["tip_rate"].replace(
    [np.inf, -np.inf],
    np.nan
)

In [ ]:

# 检查金额和衍生指标

check_columns = [
    "fare_amount",
    "total_amount",
    "tip_amount",
    "revenue_per_mile",
    "revenue_per_hour",
    "tip_rate",
]

print("\n金额及衍生指标统计：")

print(
    clean_df[check_columns]
    .describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    )
    .round(2)
    .T
)

print("\n无穷值检查：")

for column in [
    "revenue_per_mile",
    "revenue_per_hour",
    "tip_rate",
]:
    infinite_count = np.isinf(clean_df[column]).sum()
    print(f"{column}：{infinite_count:,}")

In [ ]:
# 输出清洗结果
# ============================================================

cleaned_count = len(clean_df)
deleted_count = original_count - cleaned_count
retention_rate = cleaned_count / original_count * 100

print("\n" + "=" * 70)
print("数据清洗结果")
print("=" * 70)

print(f"原始订单数：{original_count:,}")
print(f"清洗后订单数：{cleaned_count:,}")
print(f"删除订单数：{deleted_count:,}")
print(f"数据保留率：{retention_rate:.2f}%")

print("\n清洗后时间范围：")
print(
    clean_df["tpep_pickup_datetime"].min(),
    "至",
    clean_df["tpep_pickup_datetime"].max()
)

print("\n清洗后核心字段统计：")

print(
    clean_df[
        [
            "trip_distance",
            "trip_duration_minutes",
            "average_speed_mph",
            "fare_amount",
            "total_amount",
        ]
    ]
    .describe()
    .round(2)
        .T
)


In [ ]:
# 保存清洗后的数据


# 根据自己的文件夹位置修改保存路径
output_path = "yellow_taxi_2024_q1_cleaned.parquet"

# Parquet格式体积较小，且能保存字段类型
clean_df.to_parquet(
    output_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print(f"\n清洗数据已经保存至：{output_path}")

In [ ]:
clean_df.columns

# 关联出租车上下车区域

In [ ]:
# 本步骤完成：
# 1. 读取TLC出租车区域对照表；
# 2. 为订单添加上车行政区和上车区域；
# 3. 为订单添加下车行政区和下车区域；
# 4. 检查无法匹配的区域；
# 5. 生成经营分析宽表。

In [ ]:
# 读取清洗后的订单数据
import numpy as np
import pandas as pd
# 如果clean_df仍然存在于当前Python环境中，可以跳过这一行
clean_df = pd.read_parquet(
    "yellow_taxi_2024_q1_cleaned.parquet"
)


print(f"订单数：{len(clean_df):,}")
print(f"字段数：{clean_df.shape[1]}")

In [ ]:
# 读取出租车区域对照表


zone_df = pd.read_csv(r'D:\DEVELOP\Python-Project\PandasProject02\taxi_zone_lookup.csv')

print("\n区域对照表前5行：")
print(zone_df.head())

print("\n区域对照表字段：")
print(zone_df.columns.tolist())

print(f"\n区域记录数：{len(zone_df):,}")

In [ ]:
# 检查区域编号是否重复

# LocationID理论上应该唯一
duplicate_zone_count = zone_df["LocationID"].duplicated().sum()

print(
    f"\n区域编号重复数：{duplicate_zone_count:,}"
)

if duplicate_zone_count > 0:
    print("警告：区域编号存在重复，请检查区域对照表。")
else:
    print("区域编号唯一，可以进行多对一关联。")

In [ ]:
# 建立上车区域对照表


pickup_zone_df = zone_df[
    [
        "LocationID",
        "Borough",
        "Zone",
        "service_zone",
    ]
].copy()

# 修改字段名称，避免和下车区域字段混淆
pickup_zone_df = pickup_zone_df.rename(
    columns={
        "LocationID": "PULocationID",
        "Borough": "pickup_borough",
        "Zone": "pickup_zone",
        "service_zone": "pickup_service_zone",
    }
)


In [ ]:
# 关联上车区域


before_merge_count = len(clean_df)

analysis_df = clean_df.merge(
    pickup_zone_df,
    on="PULocationID",
    how="left",

    # many_to_one表示：
    # 多条订单对应一个区域编号
    validate="many_to_one",
)

after_merge_count = len(analysis_df)


print("上车区域关联结果")


print(f"关联前订单数：{before_merge_count:,}")
print(f"关联后订单数：{after_merge_count:,}")
print(
    f"无法匹配上车区域的订单："
    f"{analysis_df['pickup_zone'].isna().sum():,}"
)


In [ ]:
#建立下车区域对照表


drop_off_zone_df = zone_df[
    [
        "LocationID",
        "Borough",
        "Zone",
        "service_zone",
    ]
].copy()

drop_off_zone_df = drop_off_zone_df.rename(
    columns={
        "LocationID": "DOLocationID",
        "Borough": "drop_off_borough",
        "Zone": "drop_off_zone",
        "service_zone": "drop_off_service_zone",
    }
)

In [ ]:
# 关联下车区域

before_merge_count = len(analysis_df)

analysis_df = analysis_df.merge(
    drop_off_zone_df,
    on="DOLocationID",
    how="left",
    validate="many_to_one",
)

after_merge_count = len(analysis_df)

print("\n" + "=" * 70)
print("下车区域关联结果")
print("=" * 70)

print(f"关联前订单数：{before_merge_count:,}")
print(f"关联后订单数：{after_merge_count:,}")
print(
    f"无法匹配下车区域的订单："
    f"{analysis_df['drop_off_zone'].isna().sum():,}"
)

In [ ]:
# 构建跨行政区标识


# 判断上车和下车是否位于同一个行政区
analysis_df["is_cross_borough"] = (
    analysis_df["pickup_borough"]
    != analysis_df["drop_off_borough"]
)

# 生成完整路线名称，例如：
# Upper East Side South → Midtown Center
analysis_df["route_name"] = (
    analysis_df["pickup_zone"].astype("string")
    + " → "
    + analysis_df["drop_off_zone"].astype("string")
)

In [ ]:
# 处理异常的单笔效率指标


# 不删除订单，只为极端的单笔指标建立有效版本。
# 这些字段主要用于分布图，不用于总量计算。

analysis_df["revenue_per_mile_valid"] = (
    analysis_df["revenue_per_mile"]
    .where(
        analysis_df["revenue_per_mile"]
        .between(0, 100)
    )
)

analysis_df["revenue_per_hour_valid"] = (
    analysis_df["revenue_per_hour"]
    .where(
        analysis_df["revenue_per_hour"]
        .between(0, 500)
    )
)

# 小费率只保留0%～100%的信用卡订单
analysis_df["tip_rate_valid"] = (
    analysis_df["tip_rate"]
    .where(
        analysis_df["tip_rate"]
        .between(0, 1)
    )
)

print("\n单笔衍生指标有效数量：")

print(
    analysis_df[
        [
            "revenue_per_mile_valid",
            "revenue_per_hour_valid",
            "tip_rate_valid",
        ]
    ].count()
)


In [ ]:
# 检查关联后的主要字段


display_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_borough",
    "pickup_zone",
    "drop_off_borough",
    "drop_off_zone",
    "route_name",
    "trip_distance",
    "trip_duration_minutes",
    "total_amount",
    "is_cross_borough",
]

print("\n关联后的前10条订单：")

print(
    analysis_df[display_columns].head(10)
)

In [ ]:
# 查看各行政区上车订单数量


pickup_borough_summary = (
    analysis_df
    .groupby(
        "pickup_borough",
        dropna=False,
    )
    .agg(
        order_count=(
            "tpep_pickup_datetime",
            "size",
        ),
        total_revenue=(
            "total_amount",
            "sum",
        ),
        total_distance=(
            "trip_distance",
            "sum",
        ),
        total_duration_minutes=(
            "trip_duration_minutes",
            "sum",
        ),
    )
    .reset_index()
)

# 加权单位里程收入：
# 行政区总收入 ÷ 行政区总行驶里程
pickup_borough_summary["weighted_revenue_per_mile"] = (
    pickup_borough_summary["total_revenue"]
    / pickup_borough_summary["total_distance"]
)

# 加权单位时间收入：
# 行政区总收入 ÷ 行政区总行程小时数
pickup_borough_summary["weighted_revenue_per_hour"] = (
    pickup_borough_summary["total_revenue"]
    / (
        pickup_borough_summary[
            "total_duration_minutes"
        ] / 60
    )
)

print("\n各行政区经营情况：")

print(
    pickup_borough_summary
    .sort_values(
        "order_count",
        ascending=False,
    )
    .round(2)
)

In [ ]:
# 保存经营分析宽表


output_path = (
    "yellow_taxi_2024_q1_analysis.parquet"
)

analysis_df.to_parquet(
    output_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("\n" + "=" * 70)
print("区域关联完成")
print("=" * 70)

print(f"最终订单数：{len(analysis_df):,}")
print(f"最终字段数：{analysis_df.shape[1]}")
print(f"经营分析宽表已保存至：{output_path}")

# 4建立出租车核心经营指标体系

In [ ]:
# 本步骤完成：
# 1. 计算第一季度总体经营指标；
# 2. 计算月度经营指标；
# 3. 计算星期维度经营指标；
# 4. 计算时段维度经营指标；
# 5. 将结果保存为CSV文件。
from pathlib import Path

import pandas as pd

In [ ]:
# 读取经营分析宽表


analysis_df = pd.read_parquet(
    "yellow_taxi_2024_q1_analysis.parquet"
)


print("经营分析宽表读取完成")


print(f"订单数：{len(analysis_df):,}")
print(f"字段数：{analysis_df.shape[1]}")


In [ ]:
# 二、建立通用经营指标汇总函数


def calculate_business_metrics(data, group_columns=None):
    """
    计算出租车经营指标。

    参数
    ----------
    data : pandas.DataFrame
        出租车订单数据。

    group_columns : str或list，可选
        分组字段，例如pickup_month或weekday_name。
        如果不提供，则计算整个数据集的总体指标。

    返回
    ----------
    pandas.DataFrame
        经营指标汇总表。

    说明
    ----------
    单位里程交易额和单位时间交易额采用加权计算：

    单位里程交易额 = 总交易额 / 总里程
    单位时间交易额 = 总交易额 / 总行程小时数

    不直接计算每笔订单比率的平均值，
    避免极短订单对结果造成过度影响。
    """
    if group_columns is None:
        # 为全部数据增加一个临时分组
        working_df = data.assign(
            analysis_scope="2024年第一季度"
        )

        group_columns = ["analysis_scope"]

    elif isinstance(group_columns, str):
        # 如果只传入一个字段名称，转换成列表
        working_df = data
        group_columns = [group_columns]

    else:
        working_df = data

    summary = (
        working_df
        .groupby(
            group_columns,
            dropna=False,
            observed=True,
        )
        .agg(
            # 订单总数
            order_count=(
                "total_amount",
                "size",
            ),

            # 订单交易总额
            total_transaction_amount=(
                "total_amount",
                "sum",
            ),

            # 平均订单金额
            average_order_amount=(
                "total_amount",
                "mean",
            ),

            # 订单金额中位数
            median_order_amount=(
                "total_amount",
                "median",
            ),

            # 总行驶里程
            total_distance=(
                "trip_distance",
                "sum",
            ),

            # 平均行驶里程
            average_distance=(
                "trip_distance",
                "mean",
            ),

            # 总行程时间
            total_duration_minutes=(
                "trip_duration_minutes",
                "sum",
            ),

            # 平均行程时间
            average_duration_minutes=(
                "trip_duration_minutes",
                "mean",
            ),
        )
        .reset_index()
    )

    # 加权单位里程交易额
    summary["transaction_amount_per_mile"] = (
        summary["total_transaction_amount"]
        / summary["total_distance"]
    )

    # 加权单位时间交易额
    summary["transaction_amount_per_hour"] = (
        summary["total_transaction_amount"]
        / (
            summary["total_duration_minutes"] / 60
        )
    )

    # 订单数量占比
    summary["order_share"] = (
        summary["order_count"]
        / summary["order_count"].sum()
    )

    # 交易金额占比
    summary["transaction_amount_share"] = (
        summary["total_transaction_amount"]
        / summary["total_transaction_amount"].sum()
    )

    return summary


In [ ]:
# 总体经营指标


overall_summary = calculate_business_metrics(
    analysis_df
)

print("\n" + "=" * 70)
print("第一季度总体经营指标")
print("=" * 70)

print(
    overall_summary.round(2).T
)

In [ ]:
# 月度经营指标


# 如果宽表中没有pickup_month，则重新生成
if "pickup_month" not in analysis_df.columns:
    analysis_df["pickup_month"] = (
        analysis_df[
            "tpep_pickup_datetime"
        ].dt.month
    )

monthly_summary = calculate_business_metrics(
    analysis_df,
    "pickup_month",
)

# 按月份排序
monthly_summary = monthly_summary.sort_values(
    "pickup_month"
)

print("\n" + "=" * 70)
print("月度经营指标")
print("=" * 70)

print(
    monthly_summary.round(2)
)

In [ ]:
# 五、星期经营指标


# 确保星期编号和名称存在
if "weekday_number" not in analysis_df.columns:
    analysis_df["weekday_number"] = (
        analysis_df[
            "tpep_pickup_datetime"
        ].dt.dayofweek
    )

if "weekday_name" not in analysis_df.columns:
    analysis_df["weekday_name"] = (
        analysis_df[
            "tpep_pickup_datetime"
        ].dt.day_name()
    )

weekday_summary = calculate_business_metrics(
    analysis_df,
    [
        "weekday_number",
        "weekday_name",
    ],
)
# 根据星期一至星期日排序
weekday_summary = weekday_summary.sort_values(
    "weekday_number"
)

print("\n" + "=" * 70)
print("星期经营指标")
print("=" * 70)

print(
    weekday_summary.round(2)
)



In [ ]:
# 时段经营指标


# 如果宽表中没有时段字段，则重新生成
if "time_period" not in analysis_df.columns:
    analysis_df["pickup_hour"] = (
        analysis_df[
            "tpep_pickup_datetime"
        ].dt.hour
    )

    analysis_df["time_period"] = pd.cut(
        analysis_df["pickup_hour"],
        bins=[-1, 5, 9, 16, 19, 23],
        labels=[
            "凌晨",
            "早高峰",
            "日间",
            "晚高峰",
            "夜间",
        ],
    )

time_period_summary = calculate_business_metrics(
    analysis_df,
    "time_period",
)

print("\n" + "=" * 70)
print("时段经营指标")
print("=" * 70)

print(
    time_period_summary.round(2)
)


In [ ]:
# 按小时计算订单分布

if "pickup_hour" not in analysis_df.columns:
    analysis_df["pickup_hour"] = (
        analysis_df[
            "tpep_pickup_datetime"
        ].dt.hour
    )

hourly_summary = calculate_business_metrics(
    analysis_df,
    "pickup_hour",
)

hourly_summary = hourly_summary.sort_values(
    "pickup_hour"
)

print("\n" + "=" * 70)
print("小时经营指标")
print("=" * 70)

print(
    hourly_summary.round(2)
)


In [ ]:
# 将比例转换为百分比


summary_tables = [
    overall_summary,
    monthly_summary,
    weekday_summary,
    time_period_summary,
    hourly_summary,
]

for summary_df in summary_tables:
    summary_df["order_share_percent"] = (
        summary_df["order_share"] * 100
    ).round(2)

    summary_df[
        "transaction_amount_share_percent"
    ] = (
        summary_df[
            "transaction_amount_share"
        ] * 100
    ).round(2)


In [ ]:
# 九、保存分析结果
# ============================================================

output_directory = Path("business_results")
output_directory.mkdir(
    parents=True,
    exist_ok=True,
)
overall_summary.to_csv(
    output_directory / "overall_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

monthly_summary.to_csv(
    output_directory / "monthly_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
weekday_summary.to_csv(
    output_directory / "weekday_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

time_period_summary.to_csv(
    output_directory / "time_period_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

hourly_summary.to_csv(
    output_directory / "hourly_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("核心经营指标计算完成")

print(f"结果保存目录：{output_directory.resolve()}")

# 4.补充：修正月份、星期和时段比较口径

In [ ]:
"""


本步骤完成：
1. 计算每日经营指标；
2. 计算各月份日均订单量和日均交易额；
3. 计算各星期的日均指标；
4. 根据各时段包含的小时数计算需求强度；
5. 避免因月份天数和时段长度不同产生误判。
"""

import pandas as pd


# ============================================================
# 一、确保日期和小时字段存在
# ============================================================

analysis_df["pickup_date"] = (
    analysis_df["tpep_pickup_datetime"].dt.date
)

analysis_df["pickup_month"] = (
    analysis_df["tpep_pickup_datetime"].dt.month
)

analysis_df["pickup_hour"] = (
    analysis_df["tpep_pickup_datetime"].dt.hour
)

analysis_df["weekday_number"] = (
    analysis_df["tpep_pickup_datetime"].dt.dayofweek
)

analysis_df["weekday_name"] = (
    analysis_df["tpep_pickup_datetime"].dt.day_name()
)


# ============================================================
# 二、计算每日经营指标
# ============================================================

daily_summary = (
    analysis_df
    .groupby(
        [
            "pickup_date",
            "pickup_month",
            "weekday_number",
            "weekday_name",
        ],
        observed=True,
    )
    .agg(
        order_count=(
            "total_amount",
            "size",
        ),
        transaction_amount=(
            "total_amount",
            "sum",
        ),
        total_distance=(
            "trip_distance",
            "sum",
        ),
        total_duration_minutes=(
            "trip_duration_minutes",
            "sum",
        ),
    )
    .reset_index()
)

# 每日平均订单金额
daily_summary["average_order_amount"] = (
    daily_summary["transaction_amount"]
    / daily_summary["order_count"]
)

print("=" * 70)
print("每日经营指标样本")
print("=" * 70)

print(
    daily_summary.head().round(2)
)


# ============================================================
# 三、计算月份日均经营指标
# ============================================================

monthly_daily_summary = (
    daily_summary
    .groupby(
        "pickup_month",
        observed=True,
    )
    .agg(
        observed_days=(
            "pickup_date",
            "nunique",
        ),
        average_daily_orders=(
            "order_count",
            "mean",
        ),
        average_daily_transaction_amount=(
            "transaction_amount",
            "mean",
        ),
        average_daily_distance=(
            "total_distance",
            "mean",
        ),
        daily_order_std=(
            "order_count",
            "std",
        ),
    )
    .reset_index()
)

# 日均订单量的变异系数
# 用于观察一个月内每日订单是否稳定
monthly_daily_summary[
    "daily_order_coefficient_of_variation"
] = (
    monthly_daily_summary["daily_order_std"]
    / monthly_daily_summary["average_daily_orders"]
)

print("\n" + "=" * 70)
print("月份日均经营指标")
print("=" * 70)

print(
    monthly_daily_summary.round(2)
)


# ============================================================
# 四、计算星期日均经营指标
# ============================================================

weekday_daily_summary = (
    daily_summary
    .groupby(
        [
            "weekday_number",
            "weekday_name",
        ],
        observed=True,
    )
    .agg(
        number_of_days=(
            "pickup_date",
            "nunique",
        ),
        average_daily_orders=(
            "order_count",
            "mean",
        ),
        average_daily_transaction_amount=(
            "transaction_amount",
            "mean",
        ),
        average_daily_distance=(
            "total_distance",
            "mean",
        ),
    )
    .reset_index()
    .sort_values("weekday_number")
)

print("\n" + "=" * 70)
print("星期日均经营指标")
print("=" * 70)

print(
    weekday_daily_summary.round(2)
)


# ============================================================
# 五、定义各时段包含的小时数
# ============================================================

# 当前时段划分：
# 凌晨：0—5时，共6小时
# 早高峰：6—9时，共4小时
# 日间：10—16时，共7小时
# 晚高峰：17—19时，共3小时
# 夜间：20—23时，共4小时

period_hours = {
    "凌晨": 6,
    "早高峰": 4,
    "日间": 7,
    "晚高峰": 3,
    "夜间": 4,
}


# ============================================================
# 六、计算时段需求强度
# ============================================================

period_intensity_summary = (
    analysis_df
    .groupby(
        "time_period",
        observed=True,
    )
    .agg(
        order_count=(
            "total_amount",
            "size",
        ),
        total_transaction_amount=(
            "total_amount",
            "sum",
        ),
        number_of_days=(
            "pickup_date",
            "nunique",
        ),
        average_order_amount=(
            "total_amount",
            "mean",
        ),
        average_distance=(
            "trip_distance",
            "mean",
        ),
        average_duration_minutes=(
            "trip_duration_minutes",
            "mean",
        ),
    )
    .reset_index()
)

# 将各时段对应的小时数添加到结果中
period_intensity_summary["hours_in_period"] = (
    period_intensity_summary["time_period"]
    .astype(str)
    .map(period_hours)
)

# 每日该时段的平均订单量
period_intensity_summary["average_daily_orders"] = (
    period_intensity_summary["order_count"]
    / period_intensity_summary["number_of_days"]
)

# 每日、每小时平均订单量
# 该指标可以公平比较长度不同的时段
period_intensity_summary[
    "average_orders_per_clock_hour"
] = (
    period_intensity_summary["order_count"]
    / (
        period_intensity_summary["number_of_days"]
        * period_intensity_summary["hours_in_period"]
    )
)

# 每日、每小时平均交易额
period_intensity_summary[
    "average_transaction_amount_per_clock_hour"
] = (
    period_intensity_summary[
        "total_transaction_amount"
    ]
    / (
        period_intensity_summary["number_of_days"]
        * period_intensity_summary["hours_in_period"]
    )
)

# 根据单位时钟小时订单量排序
period_intensity_summary = (
    period_intensity_summary
    .sort_values(
        "average_orders_per_clock_hour",
        ascending=False,
    )
)

print("\n" + "=" * 70)
print("时段需求强度")
print("=" * 70)

print(
    period_intensity_summary.round(2)
)


# ============================================================
# 七、保存修正后的结果
# ============================================================

daily_summary.to_csv(
    "business_results/daily_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

monthly_daily_summary.to_csv(
    "business_results/monthly_daily_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

weekday_daily_summary.to_csv(
    "business_results/weekday_daily_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

period_intensity_summary.to_csv(
    "business_results/period_intensity_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n修正后的经营指标已保存。")

# 5.区域经营与车辆流向分析

In [ ]:
"""
本步骤完成：
1. 统计各上车区域的订单规模和交易表现；
2. 比较各区域的上车量与下车量；
3. 识别车辆潜在流入区和流出区；
4. 分析机场订单；
5. 建立区域需求—载客效率四象限；
6. 保存区域分析结果。

重要说明：
出租车订单数据没有直接记录空驶车辆数量和乘客等待时间。
因此，上下车订单差只能作为车辆调度压力的代理指标，
不能直接等同于真实的车辆供需缺口。
"""

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 一、读取经营分析宽表
# ============================================================

analysis_df = pd.read_parquet(
    "yellow_taxi_2024_q1_analysis.parquet"
)

print("=" * 70)
print("区域经营分析")
print("=" * 70)

print(f"订单数量：{len(analysis_df):,}")
print(f"字段数量：{analysis_df.shape[1]}")


# ============================================================
# 二、统计上车区域经营指标
# ============================================================

pickup_zone_summary = (
    analysis_df
    .groupby(
        [
            "PULocationID",
            "pickup_borough",
            "pickup_zone",
            "pickup_service_zone",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        # 从该区域出发的订单数量
        pickup_order_count=(
            "total_amount",
            "size",
        ),

        # 从该区域出发的订单交易总额
        total_transaction_amount=(
            "total_amount",
            "sum",
        ),

        # 平均订单金额
        average_order_amount=(
            "total_amount",
            "mean",
        ),

        # 订单金额中位数
        median_order_amount=(
            "total_amount",
            "median",
        ),

        # 总行驶里程
        total_distance=(
            "trip_distance",
            "sum",
        ),

        # 平均行驶里程
        average_distance=(
            "trip_distance",
            "mean",
        ),

        # 总载客行程时间
        total_duration_minutes=(
            "trip_duration_minutes",
            "sum",
        ),

        # 平均行程时间
        average_duration_minutes=(
            "trip_duration_minutes",
            "mean",
        ),

        # 有订单的日期数量
        observed_days=(
            "pickup_date",
            "nunique",
        ),
    )
    .reset_index()
)


# ============================================================
# 三、计算上车区域标准化指标
# ============================================================

# 日均上车订单量
pickup_zone_summary["average_daily_pickup_orders"] = (
    pickup_zone_summary["pickup_order_count"]
    / pickup_zone_summary["observed_days"]
)

# 订单数量占全部订单的比例
pickup_zone_summary["pickup_order_share"] = (
    pickup_zone_summary["pickup_order_count"]
    / pickup_zone_summary["pickup_order_count"].sum()
)

# 交易金额占全部交易金额的比例
pickup_zone_summary["transaction_amount_share"] = (
    pickup_zone_summary["total_transaction_amount"]
    / pickup_zone_summary[
        "total_transaction_amount"
    ].sum()
)

# 加权单位里程交易额
pickup_zone_summary[
    "transaction_amount_per_mile"
] = (
    pickup_zone_summary["total_transaction_amount"]
    / pickup_zone_summary["total_distance"]
)

# 加权单位载客时间交易额
# 不能解释为司机真实每小时收入
pickup_zone_summary[
    "transaction_amount_per_occupied_hour"
] = (
    pickup_zone_summary["total_transaction_amount"]
    / (
        pickup_zone_summary[
            "total_duration_minutes"
        ] / 60
    )
)


# ============================================================
# 四、统计各区域的下车订单量
# ============================================================

dropoff_zone_summary = (
    analysis_df
    .groupby(
        [
            "DOLocationID",
            "drop_off_borough",
            "drop_off_zone",
            "drop_off_service_zone",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        # 到达该区域的订单数量
        dropoff_order_count=(
            "total_amount",
            "size",
        ),
    )
    .reset_index()
)

# 将下车区域字段改成与上车区域一致，
# 方便后续按照LocationID合并
dropoff_zone_summary = (
    dropoff_zone_summary.rename(
        columns={
            "DOLocationID": "PULocationID",
            "drop_off_borough":
                "drop_off_reference_borough",
            "drop_off_zone":
                "drop_off_reference_zone",
            "drop_off_service_zone":
                "drop_off_reference_service_zone",
        }
    )
)


# ============================================================
# 五、合并上车和下车订单量
# ============================================================

zone_flow_summary = pickup_zone_summary.merge(
    dropoff_zone_summary[
        [
            "PULocationID",
            "dropoff_order_count",
        ]
    ],
    on="PULocationID",
    how="outer",
    validate="one_to_one",
)

# 没有上车或下车订单的区域填充为0
zone_flow_summary[
    [
        "pickup_order_count",
        "dropoff_order_count",
    ]
] = (
    zone_flow_summary[
        [
            "pickup_order_count",
            "dropoff_order_count",
        ]
    ]
    .fillna(0)
)


# ============================================================
# 六、计算区域订单流向差
# ============================================================

# 净上车订单量：
# 上车订单数 - 下车订单数
zone_flow_summary["net_pickup_orders"] = (
    zone_flow_summary["pickup_order_count"]
    - zone_flow_summary["dropoff_order_count"]
)

# 上下车订单比
zone_flow_summary["pickup_drop_off_ratio"] = (
    zone_flow_summary["pickup_order_count"]
    / zone_flow_summary[
        "dropoff_order_count"
    ].replace(0, np.nan)
)

# 订单流向不平衡率
#
# 绝对值越大，说明上车与下车数量越不平衡。
# 分母使用上车量和下车量之和，使不同规模区域可以比较。
zone_flow_summary["flow_imbalance_rate"] = (
    zone_flow_summary["net_pickup_orders"]
    / (
        zone_flow_summary["pickup_order_count"]
        + zone_flow_summary["dropoff_order_count"]
    )
)


# ============================================================
# 七、识别潜在调度方向
# ============================================================

def classify_flow_status(row):
    """
    根据订单流向不平衡率划分区域状态。

    大于10%：
    上车订单明显多于下车订单，
    可能需要其他区域的空车补充。

    小于-10%：
    下车订单明显多于上车订单，
    可能出现车辆聚集。

    -10%～10%：
    上下车数量相对均衡。
    """

    imbalance_rate = row["flow_imbalance_rate"]

    if pd.isna(imbalance_rate):
        return "数据不足"

    if imbalance_rate > 0.10:
        return "潜在车辆补充区"

    if imbalance_rate < -0.10:
        return "潜在车辆流出区"

    return "相对均衡区"


zone_flow_summary["flow_status"] = (
    zone_flow_summary.apply(
        classify_flow_status,
        axis=1,
    )
)


# ============================================================
# 八、设置最低订单量门槛
# ============================================================

# 小样本区域的比例指标容易产生较大波动。
# 四象限和调度分析仅纳入第一季度上车订单不少于5000条的区域。
minimum_order_count = 5000

eligible_zone_df = zone_flow_summary.loc[
    zone_flow_summary["pickup_order_count"]
    >= minimum_order_count
].copy()

print(
    f"\n满足最低订单量门槛的区域数："
    f"{len(eligible_zone_df):,}"
)


# ============================================================
# 九、建立需求—载客效率四象限
# ============================================================

# 需求指标：日均上车订单量
demand_median = eligible_zone_df[
    "average_daily_pickup_orders"
].median()

# 载客效率指标：
# 单位载客时间对应的订单交易额
efficiency_median = eligible_zone_df[
    "transaction_amount_per_occupied_hour"
].median()

print(
    f"区域日均订单量中位数："
    f"{demand_median:,.2f}"
)

print(
    f"单位载客时间交易额中位数："
    f"{efficiency_median:,.2f}"
)


def classify_zone_quadrant(row):
    """
    根据需求和载客效率划分四象限。
    """

    high_demand = (
        row["average_daily_pickup_orders"]
        >= demand_median
    )

    high_efficiency = (
        row[
            "transaction_amount_per_occupied_hour"
        ]
        >= efficiency_median
    )

    if high_demand and high_efficiency:
        return "高需求—高载客效率"

    if high_demand and not high_efficiency:
        return "高需求—低载客效率"

    if not high_demand and high_efficiency:
        return "低需求—高载客效率"

    return "低需求—低载客效率"


eligible_zone_df["zone_quadrant"] = (
    eligible_zone_df.apply(
        classify_zone_quadrant,
        axis=1,
    )
)


# ============================================================
# 十、输出热门上车区域
# ============================================================

top_pickup_zones = (
    zone_flow_summary
    .sort_values(
        "pickup_order_count",
        ascending=False,
    )
    .head(15)
)

print("\n" + "=" * 70)
print("上车订单量最高的15个区域")
print("=" * 70)

print(
    top_pickup_zones[
        [
            "pickup_borough",
            "pickup_zone",
            "pickup_service_zone",
            "pickup_order_count",
            "average_daily_pickup_orders",
            "total_transaction_amount",
            "average_order_amount",
            "average_distance",
        ]
    ].round(2)
)


# ============================================================
# 十一、输出潜在车辆补充区域
# ============================================================

potential_shortage_zones = (
    eligible_zone_df
    .loc[
        eligible_zone_df["flow_status"]
        == "潜在车辆补充区"
    ]
    .sort_values(
        "net_pickup_orders",
        ascending=False,
    )
    .head(15)
)

print("\n" + "=" * 70)
print("潜在车辆补充区域")
print("=" * 70)

print(
    potential_shortage_zones[
        [
            "pickup_borough",
            "pickup_zone",
            "pickup_order_count",
            "dropoff_order_count",
            "net_pickup_orders",
            "flow_imbalance_rate",
        ]
    ].round(3)
)


# ============================================================
# 十二、输出潜在车辆流出区域
# ============================================================

potential_surplus_zones = (
    eligible_zone_df
    .loc[
        eligible_zone_df["flow_status"]
        == "潜在车辆流出区"
    ]
    .sort_values(
        "net_pickup_orders",
        ascending=True,
    )
    .head(15)
)

print("\n" + "=" * 70)
print("潜在车辆流出区域")
print("=" * 70)

print(
    potential_surplus_zones[
        [
            "pickup_borough",
            "pickup_zone",
            "pickup_order_count",
            "dropoff_order_count",
            "net_pickup_orders",
            "flow_imbalance_rate",
        ]
    ].round(3)
)


# ============================================================
# 十三、机场订单分析
# ============================================================

# 根据service_zone和具体区域名称识别机场
airport_mask = (
    analysis_df["pickup_service_zone"]
    .isin(["Airports", "EWR"])
    | analysis_df["pickup_zone"]
    .isin(
        [
            "JFK Airport",
            "LaGuardia Airport",
            "Newark Airport",
        ]
    )
)

airport_df = analysis_df.loc[
    airport_mask
].copy()

airport_summary = (
    airport_df
    .groupby(
        [
            "pickup_borough",
            "pickup_zone",
        ],
        observed=True,
    )
    .agg(
        order_count=(
            "total_amount",
            "size",
        ),
        total_transaction_amount=(
            "total_amount",
            "sum",
        ),
        average_order_amount=(
            "total_amount",
            "mean",
        ),
        median_order_amount=(
            "total_amount",
            "median",
        ),
        average_distance=(
            "trip_distance",
            "mean",
        ),
        average_duration_minutes=(
            "trip_duration_minutes",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "order_count",
        ascending=False,
    )
)

airport_summary["order_share"] = (
    airport_summary["order_count"]
    / len(analysis_df)
)

print("\n" + "=" * 70)
print("机场上车订单分析")
print("=" * 70)

print(
    airport_summary.round(2)
)


# ============================================================
# 十四、查看四象限区域数量
# ============================================================

quadrant_summary = (
    eligible_zone_df[
        "zone_quadrant"
    ]
    .value_counts()
    .rename_axis("zone_quadrant")
    .reset_index(name="zone_count")
)

print("\n" + "=" * 70)
print("区域四象限分布")
print("=" * 70)

print(quadrant_summary)


# ============================================================
# 十五、保存区域分析结果
# ============================================================

output_directory = Path("business_results")
output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

zone_flow_summary.to_csv(
    output_directory / "zone_flow_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

eligible_zone_df.to_csv(
    output_directory / "zone_quadrant_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

top_pickup_zones.to_csv(
    output_directory / "top_pickup_zones.csv",
    index=False,
    encoding="utf-8-sig",
)

potential_shortage_zones.to_csv(
    output_directory
    / "potential_vehicle_supply_zones.csv",
    index=False,
    encoding="utf-8-sig",
)

potential_surplus_zones.to_csv(
    output_directory
    / "potential_vehicle_outflow_zones.csv",
    index=False,
    encoding="utf-8-sig",
)

airport_summary.to_csv(
    output_directory / "airport_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n区域经营与订单流向分析已完成。")

In [ ]:
"""
06：热门线路与OD流向分析

OD表示Origin-Destination，即起点—终点。

本步骤完成：
1. 统计上车区域到下车区域的线路；
2. 识别订单量最高的热门线路；
3. 分析跨行政区线路；
4. 分析机场出发线路；
5. 区分同区域短途订单与跨区域订单；
6. 保存线路分析结果。
"""

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 一、读取经营分析宽表
# ============================================================

analysis_df = pd.read_parquet(
    "yellow_taxi_2024_q1_analysis.parquet"
)

print("=" * 70)
print("热门线路与OD流向分析")
print("=" * 70)

print(f"订单数量：{len(analysis_df):,}")


# ============================================================
# 二、增加线路分类字段
# ============================================================

# 判断上车和下车是否属于同一个出租车区域
analysis_df["is_same_zone"] = (
    analysis_df["PULocationID"]
    == analysis_df["DOLocationID"]
)

# 判断是否跨行政区
analysis_df["is_cross_borough"] = (
    analysis_df["pickup_borough"]
    != analysis_df["drop_off_borough"]
)

# 建立线路名称
analysis_df["route_name"] = (
    analysis_df["pickup_zone"].astype("string")
    + " → "
    + analysis_df["drop_off_zone"].astype("string")
)


# ============================================================
# 三.汇总所有OD线路
# ============================================================

# 先计算整个研究期包含的日期数量
study_days = int(
    analysis_df["pickup_date"].nunique()
)

print(f"研究天数：{study_days}")

route_summary = (
    analysis_df
    .groupby(
        [
            "PULocationID",
            "pickup_borough",
            "pickup_zone",
            "DOLocationID",
            "drop_off_borough",
            "drop_off_zone",
            "is_same_zone",
            "is_cross_borough",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        # 线路订单数量
        order_count=(
            "total_amount",
            "size",
        ),

        # 线路交易总额
        total_transaction_amount=(
            "total_amount",
            "sum",
        ),

        # 平均订单金额
        average_order_amount=(
            "total_amount",
            "mean",
        ),

        # 订单金额中位数
        median_order_amount=(
            "total_amount",
            "median",
        ),

        # 平均行驶里程
        average_distance=(
            "trip_distance",
            "mean",
        ),

        # 平均行程时间
        average_duration_minutes=(
            "trip_duration_minutes",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# 建立线路名称
# ============================================================

route_summary["route_name"] = (
    route_summary["pickup_zone"].astype("string")
    + " → "
    + route_summary["drop_off_zone"].astype("string")
)


# ============================================================
# 计算线路日均订单量
# ============================================================

# 所有线路统一除以整个第一季度的91天
route_summary["average_daily_orders"] = (
    route_summary["order_count"]
    / study_days
)


# ============================================================
# 计算线路占比
# ============================================================

# 线路订单量占全部订单的比例
route_summary["order_share"] = (
    route_summary["order_count"]
    / len(analysis_df)
)

# 线路交易额占全部订单交易额的比例
route_summary["transaction_amount_share"] = (
    route_summary["total_transaction_amount"]
    / analysis_df["total_amount"].sum()
)


# ============================================================
# 检查结果
# ============================================================

print("\n线路汇总结果：")

print(
    route_summary[
        [
            "route_name",
            "order_count",
            "average_daily_orders",
            "average_order_amount",
            "average_distance",
            "average_duration_minutes",
        ]
    ]
    .sort_values(
        "order_count",
        ascending=False,
    )
    .head(10)
    .round(2)
)

# ============================================================
# 四、识别订单量最高的热门线路
# ============================================================

top_routes = (
    route_summary
    .sort_values(
        "order_count",
        ascending=False,
    )
    .head(20)
)

print("\n" + "=" * 70)
print("订单量最高的20条线路")
print("=" * 70)

print(
    top_routes[
        [
            "route_name",
            "pickup_borough",
            "drop_off_borough",
            "order_count",
            "average_daily_orders",
            "average_order_amount",
            "average_distance",
            "average_duration_minutes",
            "is_same_zone",
        ]
    ].round(2)
)


# ============================================================
# 五、分析同区域订单
# ============================================================

same_zone_summary = (
    analysis_df
    .groupby(
        "is_same_zone",
        observed=True,
    )
    .agg(
        order_count=(
            "total_amount",
            "size",
        ),
        total_transaction_amount=(
            "total_amount",
            "sum",
        ),
        average_order_amount=(
            "total_amount",
            "mean",
        ),
        average_distance=(
            "trip_distance",
            "mean",
        ),
        average_duration_minutes=(
            "trip_duration_minutes",
            "mean",
        ),
    )
    .reset_index()
)

same_zone_summary["order_share"] = (
    same_zone_summary["order_count"]
    / same_zone_summary["order_count"].sum()
)

print("\n" + "=" * 70)
print("同区域与跨区域订单对比")
print("=" * 70)

print(
    same_zone_summary.round(3)
)


# ============================================================
# 六、分析跨行政区线路
# ============================================================

cross_borough_routes = (
    route_summary
    .loc[
        route_summary["is_cross_borough"]
        & (
            route_summary["order_count"]
            >= 1000
        )
    ]
    .sort_values(
        "order_count",
        ascending=False,
    )
    .head(20)
)

print("\n" + "=" * 70)
print("热门跨行政区线路")
print("=" * 70)

print(
    cross_borough_routes[
        [
            "route_name",
            "pickup_borough",
            "drop_off_borough",
            "order_count",
            "average_order_amount",
            "average_distance",
            "average_duration_minutes",
        ]
    ].round(2)
)


# ============================================================
# 七、分析JFK出发线路
# ============================================================

jfk_routes = (
    route_summary
    .loc[
        route_summary["pickup_zone"]
        == "JFK Airport"
    ]
    .sort_values(
        "order_count",
        ascending=False,
    )
    .head(15)
)

print("\n" + "=" * 70)
print("JFK出发热门线路")
print("=" * 70)

print(
    jfk_routes[
        [
            "route_name",
            "drop_off_borough",
            "order_count",
            "average_daily_orders",
            "average_order_amount",
            "average_distance",
            "average_duration_minutes",
        ]
    ].round(2)
)


# ============================================================
# 八、分析LaGuardia出发线路
# ============================================================

laguardia_routes = (
    route_summary
    .loc[
        route_summary["pickup_zone"]
        == "LaGuardia Airport"
    ]
    .sort_values(
        "order_count",
        ascending=False,
    )
    .head(15)
)

print("\n" + "=" * 70)
print("LaGuardia出发热门线路")
print("=" * 70)

print(
    laguardia_routes[
        [
            "route_name",
            "drop_off_borough",
            "order_count",
            "average_daily_orders",
            "average_order_amount",
            "average_distance",
            "average_duration_minutes",
        ]
    ].round(2)
)


# ============================================================
# 九、汇总行政区之间的订单流向
# ============================================================

borough_flow_summary = (
    analysis_df
    .groupby(
        [
            "pickup_borough",
            "drop_off_borough",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        order_count=(
            "total_amount",
            "size",
        ),
        total_transaction_amount=(
            "total_amount",
            "sum",
        ),
        average_order_amount=(
            "total_amount",
            "mean",
        ),
        average_distance=(
            "trip_distance",
            "mean",
        ),
        average_duration_minutes=(
            "trip_duration_minutes",
            "mean",
        ),
    )
    .reset_index()
)

borough_flow_summary["order_share"] = (
    borough_flow_summary["order_count"]
    / borough_flow_summary["order_count"].sum()
)

borough_flow_summary = (
    borough_flow_summary.sort_values(
        "order_count",
        ascending=False,
    )
)

print("\n" + "=" * 70)
print("行政区之间的订单流向")
print("=" * 70)

print(
    borough_flow_summary.head(20).round(3)
)


# ============================================================
# 十、为高价值线路设置最低样本量
# ============================================================

# 订单过少的线路即使平均金额很高，
# 也可能只是少数异常或偶然订单。
#
# 因此只在订单量不少于1000条的线路中
# 筛选平均订单金额较高的线路。

high_value_routes = (
    route_summary
    .loc[
        route_summary["order_count"] >= 1000
    ]
    .sort_values(
        "average_order_amount",
        ascending=False,
    )
    .head(20)
)

print("\n" + "=" * 70)
print("高交易金额线路")
print("=" * 70)

print(
    high_value_routes[
        [
            "route_name",
            "order_count",
            "average_order_amount",
            "median_order_amount",
            "average_distance",
            "average_duration_minutes",
        ]
    ].round(2)
)


# ============================================================
# 十一、检查线路订单量分布
# ============================================================

print("\n" + "=" * 70)
print("线路订单量分布")
print("=" * 70)

print(
    route_summary["order_count"]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .round(2)
)


# ============================================================
# 十二、保存分析结果
# ============================================================

output_directory = Path("business_results")

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

route_summary.to_csv(
    output_directory / "route_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

top_routes.to_csv(
    output_directory / "top_routes.csv",
    index=False,
    encoding="utf-8-sig",
)

cross_borough_routes.to_csv(
    output_directory / "cross_borough_routes.csv",
    index=False,
    encoding="utf-8-sig",
)

jfk_routes.to_csv(
    output_directory / "jfk_routes.csv",
    index=False,
    encoding="utf-8-sig",
)

laguardia_routes.to_csv(
    output_directory / "laguardia_routes.csv",
    index=False,
    encoding="utf-8-sig",
)

borough_flow_summary.to_csv(
    output_directory / "borough_flow_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

high_value_routes.to_csv(
    output_directory / "high_value_routes.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n热门线路与OD流向分析已完成。")

# 7.核心经营分析可视化

In [ ]:
#pip install matplotlib seaborn

In [ ]:
"""
纽约出租车经营分析项目
步骤07：核心经营分析可视化（优化版）

生成图表：
1. 月度日均订单量与日均交易额
2. 星期日均订单需求
3. 不同时段的每小时订单需求强度
4. 上车订单量最高的15个区域
5. 订单量最高的10条OD线路

数据来源：
NYC TLC Trip Record Data，2024年第一季度
"""

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib import font_manager
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter


# ============================================================
# 一、设置文件路径
# ============================================================

# 上一步保存的CSV汇总表目录
RESULT_DIR = Path("business_results")

# 图片输出目录
FIGURE_DIR = Path("business_figures")

# 如果目录不存在，则自动创建
FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"分析结果目录：{RESULT_DIR.resolve()}")
print(f"图片保存目录：{FIGURE_DIR.resolve()}")


# ============================================================
# 二、配置中文字体
# ============================================================

def configure_chinese_font():
    """
    自动查找Windows系统中的常用中文字体。
    """

    preferred_fonts = [
        "Microsoft YaHei",
        "SimHei",
        "SimSun",
        "Arial Unicode MS",
    ]

    installed_fonts = {
        font.name
        for font in font_manager.fontManager.ttflist
    }

    for font_name in preferred_fonts:
        if font_name in installed_fonts:
            plt.rcParams["font.sans-serif"] = [
                font_name
            ]

            print(f"使用中文字体：{font_name}")
            break
    else:
        print(
            "未找到常用中文字体，"
            "请检查图表中的中文是否正常显示。"
        )

    # 防止坐标轴负号显示为方框
    plt.rcParams["axes.unicode_minus"] = False


# 设置Seaborn基础风格
sns.set_theme(
    style="whitegrid",
    context="notebook",
)

# Seaborn可能覆盖字体，因此在设置主题后配置字体
configure_chinese_font()


# ============================================================
# 三、设置统一颜色
# ============================================================

# 普通指标颜色
BLUE = "#4C84B1"

# 次要蓝色
LIGHT_BLUE = "#8FB7D8"

# 强调色
ORANGE = "#E97842"

# 机场颜色
AIRPORT_ORANGE = "#EE8A36"

# 辅助文字颜色
GRAY = "#777777"

# 网格线颜色
GRID_COLOR = "#E5E5E5"


# ============================================================
# 四、设置统一数据来源
# ============================================================

SOURCE_NOTE = (
    "数据来源：NYC TLC Trip Record Data，"
    "2024年第一季度"
)


def add_source_note(fig):
    """
    在图片右下角添加统一的数据来源。
    """

    fig.text(
        0.99,
        0.012,
        SOURCE_NOTE,
        ha="right",
        va="bottom",
        fontsize=8,
        color=GRAY,
    )


def format_integer_axis(value, position):
    """
    坐标轴整数增加千位分隔符。
    例如：100000显示为100,000。
    """

    return f"{value:,.0f}"


# ============================================================
# 五、读取汇总数据
# ============================================================

monthly_df = pd.read_csv(
    RESULT_DIR / "monthly_daily_summary.csv"
)

weekday_df = pd.read_csv(
    RESULT_DIR / "weekday_daily_summary.csv"
)

period_df = pd.read_csv(
    RESULT_DIR / "period_intensity_summary.csv"
)

top_zone_df = pd.read_csv(
    RESULT_DIR / "top_pickup_zones.csv"
)

top_route_df = pd.read_csv(
    RESULT_DIR / "top_routes.csv"
)

print("经营分析汇总数据读取完成。")


# ============================================================
# 六、月度日均经营趋势图
# ============================================================

month_mapping = {
    1: "1月",
    2: "2月",
    3: "3月",
}

monthly_df["month_label"] = (
    monthly_df["pickup_month"]
    .map(month_mapping)
)

# 将日均交易额转换为万美元
monthly_df["average_daily_amount_10k"] = (
    monthly_df[
        "average_daily_transaction_amount"
    ] / 10000
)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12, 5.5),
)

# ------------------------------------------------------------
# 左图：月度日均订单量
# ------------------------------------------------------------

bars_left = axes[0].bar(
    monthly_df["month_label"],
    monthly_df["average_daily_orders"],
    color=BLUE,
    width=0.72,
)

axes[0].set_title(
    "月度日均订单量",
    fontsize=15,
    fontweight="bold",
    pad=12,
)

axes[0].set_xlabel("")
axes[0].set_ylabel("日均订单量（条）")

axes[0].yaxis.set_major_formatter(
    FuncFormatter(format_integer_axis)
)

axes[0].bar_label(
    bars_left,
    labels=[
        f"{value:,.0f}"
        for value in monthly_df[
            "average_daily_orders"
        ]
    ],
    padding=4,
    fontsize=10,
)

# 为柱顶数字预留空间
axes[0].set_ylim(
    0,
    monthly_df[
        "average_daily_orders"
    ].max() * 1.12,
)

axes[0].grid(
    axis="x",
    visible=False,
)


# ------------------------------------------------------------
# 右图：月度日均交易额
# ------------------------------------------------------------

bars_right = axes[1].bar(
    monthly_df["month_label"],
    monthly_df["average_daily_amount_10k"],
    color=ORANGE,
    width=0.72,
)

axes[1].set_title(
    "月度日均订单交易额",
    fontsize=15,
    fontweight="bold",
    pad=12,
)

axes[1].set_xlabel("")
axes[1].set_ylabel("日均订单交易额（万美元）")

axes[1].bar_label(
    bars_right,
    labels=[
        f"{value:,.1f}"
        for value in monthly_df[
            "average_daily_amount_10k"
        ]
    ],
    padding=4,
    fontsize=10,
)

axes[1].set_ylim(
    0,
    monthly_df[
        "average_daily_amount_10k"
    ].max() * 1.12,
)

axes[1].grid(
    axis="x",
    visible=False,
)

# 删除上方和右侧边框
sns.despine(
    ax=axes[0],
)

sns.despine(
    ax=axes[1],
)

fig.suptitle(
    "2024年第一季度出租车月度经营趋势",
    fontsize=17,
    fontweight="bold",
    y=1.01,
)

add_source_note(fig)

fig.tight_layout(
    rect=[0, 0.05, 1, 0.96]
)

monthly_output_path = (
    FIGURE_DIR
    / "01_monthly_daily_business_optimized.png"
)

fig.savefig(
    monthly_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 七、星期日均订单需求图
# ============================================================

weekday_chinese = {
    "Monday": "星期一",
    "Tuesday": "星期二",
    "Wednesday": "星期三",
    "Thursday": "星期四",
    "Friday": "星期五",
    "Saturday": "星期六",
    "Sunday": "星期日",
}

weekday_df["weekday_label"] = (
    weekday_df["weekday_name"]
    .map(weekday_chinese)
)

weekday_df = weekday_df.sort_values(
    "weekday_number"
).copy()

# 普通日期统一使用浅蓝色，
# 仅突出订单量最高的星期四
weekday_colors = [
    ORANGE if day == "星期四"
    else LIGHT_BLUE
    for day in weekday_df["weekday_label"]
]

fig, ax = plt.subplots(
    figsize=(10.5, 5.8)
)

bars = ax.bar(
    weekday_df["weekday_label"],
    weekday_df["average_daily_orders"],
    color=weekday_colors,
    width=0.72,
)

ax.set_title(
    "不同星期的日均订单需求",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel("")
ax.set_ylabel("日均订单量（条）")

ax.yaxis.set_major_formatter(
    FuncFormatter(format_integer_axis)
)

ax.bar_label(
    bars,
    labels=[
        f"{value:,.0f}"
        for value in weekday_df[
            "average_daily_orders"
        ]
    ],
    padding=4,
    fontsize=9,
)

ax.set_ylim(
    0,
    weekday_df[
        "average_daily_orders"
    ].max() * 1.12,
)

ax.grid(
    axis="x",
    visible=False,
)

# 增加颜色说明
weekday_legend = [
    Patch(
        facecolor=ORANGE,
        label="日均订单量最高",
    ),
    Patch(
        facecolor=LIGHT_BLUE,
        label="其他星期",
    ),
]

ax.legend(
    handles=weekday_legend,
    loc="upper left",
    frameon=False,
)

sns.despine(ax=ax)

add_source_note(fig)

fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)

weekday_output_path = (
    FIGURE_DIR
    / "02_weekday_demand_optimized.png"
)

fig.savefig(
    weekday_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 八、不同时段需求强度图
# ============================================================

# 按自然时间顺序排列
period_order = [
    "凌晨",
    "早高峰",
    "日间",
    "晚高峰",
    "夜间",
]

period_df["time_period"] = pd.Categorical(
    period_df["time_period"],
    categories=period_order,
    ordered=True,
)

period_df = period_df.sort_values(
    "time_period"
).copy()

# 普通时段统一使用浅蓝色，
# 只突出需求强度最高的晚高峰
period_colors = [
    ORANGE if period == "晚高峰"
    else LIGHT_BLUE
    for period in period_df[
        "time_period"
    ].astype(str)
]

fig, ax = plt.subplots(
    figsize=(9.5, 5.8)
)

bars = ax.bar(
    period_df["time_period"].astype(str),
    period_df[
        "average_orders_per_clock_hour"
    ],
    color=period_colors,
    width=0.68,
)

ax.set_title(
    "不同时段的每小时订单需求强度",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

# 修正原来的“上午时段”
ax.set_xlabel("上车时段")
ax.set_ylabel("每日每小时平均订单量（条）")

ax.yaxis.set_major_formatter(
    FuncFormatter(format_integer_axis)
)

ax.bar_label(
    bars,
    labels=[
        f"{value:,.0f}"
        for value in period_df[
            "average_orders_per_clock_hour"
        ]
    ],
    padding=4,
    fontsize=10,
)

ax.set_ylim(
    0,
    period_df[
        "average_orders_per_clock_hour"
    ].max() * 1.13,
)

ax.grid(
    axis="x",
    visible=False,
)

period_legend = [
    Patch(
        facecolor=ORANGE,
        label="需求强度最高",
    ),
    Patch(
        facecolor=LIGHT_BLUE,
        label="其他时段",
    ),
]

ax.legend(
    handles=period_legend,
    loc="upper left",
    frameon=False,
)

sns.despine(ax=ax)

add_source_note(fig)

fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)

period_output_path = (
    FIGURE_DIR
    / "03_time_period_demand_optimized.png"
)

fig.savefig(
    period_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 九、热门上车区域图
# ============================================================

# 取订单量最高的15个上车区域
zone_plot_df = (
    top_zone_df
    .head(15)
    .sort_values(
        "pickup_order_count",
        ascending=True,
    )
    .copy()
)

# 转换为万条
zone_plot_df["order_count_10k"] = (
    zone_plot_df["pickup_order_count"]
    / 10000
)

# 机场使用橙色，普通区域使用蓝色
zone_colors = [
    AIRPORT_ORANGE
    if service_zone == "Airports"
    else BLUE
    for service_zone in zone_plot_df[
        "pickup_service_zone"
    ]
]

fig, ax = plt.subplots(
    figsize=(11, 7.5)
)

bars = ax.barh(
    zone_plot_df["pickup_zone"],
    zone_plot_df["order_count_10k"],
    color=zone_colors,
    height=0.72,
)

ax.set_title(
    "上车订单量最高的15个区域",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel("第一季度上车订单量（万条）")
ax.set_ylabel("")

ax.bar_label(
    bars,
    labels=[
        f"{value:.1f}"
        for value in zone_plot_df[
            "order_count_10k"
        ]
    ],
    padding=4,
    fontsize=9,
)

ax.set_xlim(
    0,
    zone_plot_df[
        "order_count_10k"
    ].max() * 1.12,
)

ax.grid(
    axis="y",
    visible=False,
)

zone_legend = [
    Patch(
        facecolor=BLUE,
        label="普通出租车区域",
    ),
    Patch(
        facecolor=AIRPORT_ORANGE,
        label="机场区域",
    ),
]

ax.legend(
    handles=zone_legend,
    loc="lower right",
    frameon=False,
)

sns.despine(ax=ax)

add_source_note(fig)

fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)

zone_output_path = (
    FIGURE_DIR
    / "04_top_pickup_zones_optimized.png"
)

fig.savefig(
    zone_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 十、热门OD线路图
# ============================================================

# 正文只展示前10条，避免线路名称过于拥挤
route_plot_df = (
    top_route_df
    .head(10)
    .sort_values(
        "order_count",
        ascending=True,
    )
    .copy()
)

fig, ax = plt.subplots(
    figsize=(12.5, 7)
)

bars = ax.barh(
    route_plot_df["route_name"],
    route_plot_df["order_count"],
    color=BLUE,
    height=0.70,
)

ax.set_title(
    "订单量最高的10条上车—下车线路",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel("第一季度订单量（条）")
ax.set_ylabel("")

ax.xaxis.set_major_formatter(
    FuncFormatter(format_integer_axis)
)

ax.bar_label(
    bars,
    labels=[
        f"{value:,.0f}"
        for value in route_plot_df[
            "order_count"
        ]
    ],
    padding=4,
    fontsize=9,
)

ax.set_xlim(
    0,
    route_plot_df[
        "order_count"
    ].max() * 1.13,
)

ax.grid(
    axis="y",
    visible=False,
)

sns.despine(ax=ax)

add_source_note(fig)

# 为左侧较长的线路名称预留空间
fig.subplots_adjust(
    left=0.40,
    right=0.97,
    top=0.90,
    bottom=0.14,
)

route_output_path = (
    FIGURE_DIR
    / "05_top_od_routes_optimized.png"
)

fig.savefig(
    route_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 十一、检查图片是否全部保存
# ============================================================

output_files = [
    monthly_output_path,
    weekday_output_path,
    period_output_path,
    zone_output_path,
    route_output_path,
]

print("\n" + "=" * 70)
print("优化版经营分析图表生成完成")
print("=" * 70)

for file_path in output_files:
    print(
        f"{file_path.name}："
        f"{'保存成功' if file_path.exists() else '保存失败'}"
    )

print(f"\n图片保存目录：{FIGURE_DIR.resolve()}")

In [ ]:
"""
纽约出租车经营分析项目
步骤08：区域与订单流向高级可视化

生成图表：
1. 区域需求—载客效率四象限
2. 区域订单流向不平衡
3. 机场与非机场订单价值对比
4. 行政区OD流向热力图

说明：
“载客效率”表示每个载客行程小时对应的订单交易额，
不等于司机实际每小时收入。
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import (
    FuncFormatter,
    PercentFormatter,
)

# ==============================================================
# 统一配色
# ==============================================================

BLUE = "#4C83B6"
LIGHT_BLUE = "#91B9D8"
ORANGE = "#EF7B3A"
AIRPORT_ORANGE = "#EF7B3A"  # 机场订单强调色
GREEN = "#6DB18C"
PURPLE = "#9182B4"
GRAY = "#777777"
DARK_GRAY = "#2B2B2B"


# ============================================================
# 一、设置文件路径
# ============================================================

RESULT_DIR = Path("business_results")
FIGURE_DIR = Path("business_figures")

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"数据目录：{RESULT_DIR.resolve()}")
print(f"图片目录：{FIGURE_DIR.resolve()}")


# ============================================================
# 二、配置中文字体
# ============================================================

def configure_chinese_font():
    """
    自动选择Windows中常见的中文字体。
    """

    preferred_fonts = [
        "Microsoft YaHei",
        "SimHei",
        "SimSun",
        "Arial Unicode MS",
    ]

    installed_fonts = {
        font.name
        for font in font_manager.fontManager.ttflist
    }

    for font_name in preferred_fonts:
        if font_name in installed_fonts:
            plt.rcParams["font.sans-serif"] = [
                font_name
            ]

            print(f"使用中文字体：{font_name}")
            break
    else:
        print("未找到常见中文字体。")

    plt.rcParams["axes.unicode_minus"] = False


sns.set_theme(
    style="whitegrid",
    context="notebook",
)

configure_chinese_font()


# ============================================================
# 三、统一颜色和数据来源
# ============================================================

BLUE = "#4C84B1"
LIGHT_BLUE = "#8FB7D8"
ORANGE = "#E97842"
GREEN = "#58A27C"
PURPLE = "#8675A9"
GRAY = "#777777"
LIGHT_GRAY = "#D9D9D9"

SOURCE_NOTE = (
    "数据来源：NYC TLC Trip Record Data，"
    "2024年第一季度"
)


def add_source_note(fig, extra_note=None):
    """
    在图片下方添加数据来源和必要说明。
    """

    note = SOURCE_NOTE

    if extra_note:
        note = f"{note}；{extra_note}"

    fig.text(
        0.99,
        0.012,
        note,
        ha="right",
        va="bottom",
        fontsize=8,
        color=GRAY,
    )


def integer_formatter(value, position):
    """
    为整数坐标增加千位分隔符。
    """

    return f"{value:,.0f}"


# ============================================================
# 四、读取分析结果
# ============================================================

zone_df = pd.read_csv(
    RESULT_DIR / "zone_quadrant_summary.csv"
)

zone_flow_df = pd.read_csv(
    RESULT_DIR / "zone_flow_summary.csv"
)

airport_df = pd.read_csv(
    RESULT_DIR / "airport_summary.csv"
)

overall_df = pd.read_csv(
    RESULT_DIR / "overall_summary.csv"
)

borough_flow_df = pd.read_csv(
    RESULT_DIR / "borough_flow_summary.csv"
)

print("高级可视化所需数据读取完成。")


# ============================================================
# 五、区域需求—载客效率四象限
# ============================================================

# 重新计算四象限分割线
demand_median = zone_df[
    "average_daily_pickup_orders"
].median()

efficiency_median = zone_df[
    "transaction_amount_per_occupied_hour"
].median()

# 设置四象限颜色
quadrant_colors = {
    "高需求—高载客效率": GREEN,
    "高需求—低载客效率": ORANGE,
    "低需求—高载客效率": PURPLE,
    "低需求—低载客效率": LIGHT_BLUE,
}

zone_df["point_color"] = (
    zone_df["zone_quadrant"]
    .map(quadrant_colors)
)

# 气泡大小根据区域交易总额确定
# 使用平方根压缩极端差异
transaction_sqrt = np.sqrt(
    zone_df["total_transaction_amount"]
)

zone_df["bubble_size"] = (
    100
    + (
        transaction_sqrt
        - transaction_sqrt.min()
    )
    / (
        transaction_sqrt.max()
        - transaction_sqrt.min()
    )
    * 700
)

fig, ax = plt.subplots(
    figsize=(11, 7.5)
)

# 按四象限分别绘制，便于生成图例
for quadrant_name, color in quadrant_colors.items():

    plot_data = zone_df.loc[
        zone_df["zone_quadrant"]
        == quadrant_name
    ]

    ax.scatter(
        plot_data["average_daily_pickup_orders"],
        plot_data[
            "transaction_amount_per_occupied_hour"
        ],
        s=plot_data["bubble_size"],
        color=color,
        alpha=0.72,
        edgecolor="white",
        linewidth=0.8,
        label=quadrant_name,
    )


# 绘制需求中位数线
ax.axvline(
    demand_median,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
)

# 绘制载客效率中位数线
ax.axhline(
    efficiency_median,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
)


# ------------------------------------------------------------
# 标注重点区域
# ------------------------------------------------------------

# 标注订单量最高的6个区域
label_zone_df = (
    zone_df
    .nlargest(
        6,
        "pickup_order_count",
    )
    .copy()
)

# 确保两个机场都被标注
airport_zone_labels = zone_df.loc[
    zone_df["pickup_zone"].isin(
        [
            "JFK Airport",
            "LaGuardia Airport",
        ]
    )
]

label_zone_df = (
    pd.concat(
        [
            label_zone_df,
            airport_zone_labels,
        ]
    )
    .drop_duplicates(
        subset="PULocationID"
    )
)

# 为不同区域设置不同偏移，尽量避免文字重叠
annotation_offsets = [
    (8, 8),
    (8, -14),
    (-80, 10),
    (8, 10),
    (8, -15),
    (-90, -12),
    (10, 14),
    (-90, 14),
]

for index, (_, row) in enumerate(
    label_zone_df.iterrows()
):
    offset = annotation_offsets[
        index % len(annotation_offsets)
    ]

    ax.annotate(
        row["pickup_zone"],
        xy=(
            row["average_daily_pickup_orders"],
            row[
                "transaction_amount_per_occupied_hour"
            ],
        ),
        xytext=offset,
        textcoords="offset points",
        fontsize=8.5,
        color="#333333",
        arrowprops={
            "arrowstyle": "-",
            "color": "#999999",
            "linewidth": 0.7,
        },
    )


ax.set_title(
    "出租车区域需求—载客效率四象限",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel("日均上车订单量（条）")

ax.set_ylabel(
    "单位载客时间订单交易额（美元/小时）"
)

ax.xaxis.set_major_formatter(
    FuncFormatter(integer_formatter)
)

ax.legend(
    title="区域类型",
    loc="best",
    frameon=False,
    fontsize=9,
    title_fontsize=10,
)

ax.grid(
    color="#E5E5E5",
    linewidth=0.8,
)

sns.despine(ax=ax)

add_source_note(
    fig,
    "气泡大小表示区域订单交易总额",
)

fig.tight_layout(
    rect=[0, 0.055, 1, 1]
)

quadrant_output_path = (
    FIGURE_DIR
    / "06_zone_demand_efficiency_quadrant.png"
)

fig.savefig(
    quadrant_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 六、区域订单流向不平衡图
# ============================================================

# 仅保留订单量不少于5000条的区域
eligible_flow_df = zone_flow_df.loc[
    zone_flow_df["pickup_order_count"]
    >= 5000
].copy()

# 上车明显多于下车的前8个区域
positive_flow_df = (
    eligible_flow_df
    .loc[
        eligible_flow_df["net_pickup_orders"] > 0
    ]
    .nlargest(
        8,
        "net_pickup_orders",
    )
)

# 下车明显多于上车的前8个区域
negative_flow_df = (
    eligible_flow_df
    .loc[
        eligible_flow_df["net_pickup_orders"] < 0
    ]
    .nsmallest(
        8,
        "net_pickup_orders",
    )
)

flow_plot_df = pd.concat(
    [
        negative_flow_df,
        positive_flow_df,
    ],
    ignore_index=True,
)

# 转换为百分比
flow_plot_df["flow_imbalance_percent"] = (
    flow_plot_df["flow_imbalance_rate"]
    * 100
)

# 按不平衡率排序
flow_plot_df = flow_plot_df.sort_values(
    "flow_imbalance_percent"
)

# 正值为橙色，负值为蓝色
flow_colors = [
    ORANGE if value > 0
    else BLUE
    for value in flow_plot_df[
        "flow_imbalance_percent"
    ]
]

fig, ax = plt.subplots(
    figsize=(11, 8)
)

bars = ax.barh(
    flow_plot_df["pickup_zone"],
    flow_plot_df["flow_imbalance_percent"],
    color=flow_colors,
    height=0.70,
)

# 零值参考线
ax.axvline(
    0,
    color="#444444",
    linewidth=1,
)

ax.set_title(
    "重点区域上车—下车订单流向不平衡",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel(
    "订单流向不平衡率"
)

ax.set_ylabel("")

ax.xaxis.set_major_formatter(
    PercentFormatter(
        xmax=100,
        decimals=0,
    )
)

# 在柱形末端显示净订单数量
for bar, net_orders in zip(
    bars,
    flow_plot_df["net_pickup_orders"],
):

    bar_width = bar.get_width()
    y_position = (
        bar.get_y()
        + bar.get_height() / 2
    )

    label_text = (
        f"{net_orders:+,.0f}条"
    )

    if bar_width >= 0:
        text_x = bar_width + 1
        horizontal_alignment = "left"
    else:
        text_x = bar_width - 1
        horizontal_alignment = "right"

    ax.text(
        text_x,
        y_position,
        label_text,
        va="center",
        ha=horizontal_alignment,
        fontsize=8.5,
    )

# 为标签预留空间
maximum_absolute_rate = (
    flow_plot_df[
        "flow_imbalance_percent"
    ]
    .abs()
    .max()
)

ax.set_xlim(
    -maximum_absolute_rate * 1.35,
    maximum_absolute_rate * 1.35,
)

flow_legend = [
    Patch(
        facecolor=ORANGE,
        label="上车量高于下车量",
    ),
    Patch(
        facecolor=BLUE,
        label="下车量高于上车量",
    ),
]

ax.legend(
    handles=flow_legend,
    loc="lower right",
    frameon=False,
)

ax.grid(
    axis="y",
    visible=False,
)

sns.despine(ax=ax)

add_source_note(
    fig,
    "该指标仅作为车辆再平衡压力的代理变量",
)

fig.tight_layout(
    rect=[0, 0.055, 1, 1]
)

flow_output_path = (
    FIGURE_DIR
    / "07_zone_flow_imbalance.png"
)

fig.savefig(
    flow_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 七、机场与非机场订单价值对比
# ============================================================

# 提取总体指标
overall_orders = float(
    overall_df["order_count"].iloc[0]
)

overall_amount = float(
    overall_df[
        "total_transaction_amount"
    ].iloc[0]
)

# 机场订单汇总
airport_orders = float(
    airport_df["order_count"].sum()
)

airport_amount = float(
    airport_df[
        "total_transaction_amount"
    ].sum()
)

# 非机场订单
non_airport_orders = (
    overall_orders - airport_orders
)

non_airport_amount = (
    overall_amount - airport_amount
)

airport_compare_df = pd.DataFrame(
    {
        "group": [
            "机场上车订单",
            "非机场上车订单",
        ],
        "order_count": [
            airport_orders,
            non_airport_orders,
        ],
        "transaction_amount": [
            airport_amount,
            non_airport_amount,
        ],
    }
)

# 订单数量占比
airport_compare_df["order_share"] = (
    airport_compare_df["order_count"]
    / overall_orders
    * 100
)

# 交易金额占比
airport_compare_df["amount_share"] = (
    airport_compare_df[
        "transaction_amount"
    ]
    / overall_amount
    * 100
)

# 平均订单金额
airport_compare_df[
    "average_order_amount"
] = (
    airport_compare_df[
        "transaction_amount"
    ]
    / airport_compare_df["order_count"]
)


fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12, 5.8),
)

group_colors = [
    AIRPORT_ORANGE,
    BLUE,
]


# ------------------------------------------------------------
# 左图：订单和交易金额占比
# ------------------------------------------------------------

x_positions = np.arange(
    len(airport_compare_df)
)

bar_width = 0.34

order_share_bars = axes[0].bar(
    x_positions - bar_width / 2,
    airport_compare_df["order_share"],
    width=bar_width,
    color=LIGHT_BLUE,
    label="订单量占比",
)

amount_share_bars = axes[0].bar(
    x_positions + bar_width / 2,
    airport_compare_df["amount_share"],
    width=bar_width,
    color=ORANGE,
    label="交易金额占比",
)

axes[0].set_xticks(
    x_positions
)

axes[0].set_xticklabels(
    airport_compare_df["group"]
)

axes[0].set_title(
    "机场订单数量与交易金额贡献",
    fontsize=14,
    fontweight="bold",
)

axes[0].set_ylabel("占全部订单比例")

axes[0].yaxis.set_major_formatter(
    PercentFormatter(
        xmax=100,
        decimals=0,
    )
)

axes[0].bar_label(
    order_share_bars,
    labels=[
        f"{value:.1f}%"
        for value in airport_compare_df[
            "order_share"
        ]
    ],
    padding=3,
    fontsize=9,
)

axes[0].bar_label(
    amount_share_bars,
    labels=[
        f"{value:.1f}%"
        for value in airport_compare_df[
            "amount_share"
        ]
    ],
    padding=3,
    fontsize=9,
)

axes[0].set_ylim(
    0,
    max(
        airport_compare_df["order_share"].max(),
        airport_compare_df["amount_share"].max(),
    ) * 1.12,
)

axes[0].legend(
    frameon=False,
)

axes[0].grid(
    axis="x",
    visible=False,
)


# ------------------------------------------------------------
# 右图：平均订单金额
# ------------------------------------------------------------

average_amount_bars = axes[1].bar(
    airport_compare_df["group"],
    airport_compare_df[
        "average_order_amount"
    ],
    color=group_colors,
    width=0.62,
)

axes[1].set_title(
    "机场与非机场平均订单金额",
    fontsize=14,
    fontweight="bold",
)

axes[1].set_ylabel("平均订单金额（美元）")

axes[1].bar_label(
    average_amount_bars,
    labels=[
        f"${value:.2f}"
        for value in airport_compare_df[
            "average_order_amount"
        ]
    ],
    padding=4,
    fontsize=10,
)

axes[1].set_ylim(
    0,
    airport_compare_df[
        "average_order_amount"
    ].max() * 1.15,
)

axes[1].grid(
    axis="x",
    visible=False,
)

sns.despine(
    ax=axes[0],
)

sns.despine(
    ax=axes[1],
)

fig.suptitle(
    "机场订单价值特征",
    fontsize=17,
    fontweight="bold",
    y=1.01,
)

add_source_note(fig)

fig.tight_layout(
    rect=[0, 0.05, 1, 0.96]
)

airport_output_path = (
    FIGURE_DIR
    / "08_airport_value_comparison.png"
)

fig.savefig(
    airport_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 八、行政区OD流向热力图
# ============================================================

# 兼容不同的下车行政区字段名称
possible_dropoff_columns = [
    "dropoff_borough",
    "drop_off_borough",
]

dropoff_borough_column = None

for column in possible_dropoff_columns:
    if column in borough_flow_df.columns:
        dropoff_borough_column = column
        break

if dropoff_borough_column is None:
    raise KeyError(
        "未找到下车行政区字段，"
        "请检查borough_flow_summary.csv。"
    )


# 建立OD订单量矩阵
borough_od_matrix = (
    borough_flow_df
    .pivot_table(
        index="pickup_borough",
        columns=dropoff_borough_column,
        values="order_count",
        aggfunc="sum",
        fill_value=0,
    )
)

# 设置行政区展示顺序
borough_order = [
    "Manhattan",
    "Queens",
    "Brooklyn",
    "Bronx",
    "Staten Island",
    "EWR",
]

# 只保留实际出现的行政区
row_order = [
    borough
    for borough in borough_order
    if borough in borough_od_matrix.index
]

column_order = [
    borough
    for borough in borough_order
    if borough in borough_od_matrix.columns
]

borough_od_matrix = (
    borough_od_matrix
    .reindex(
        index=row_order,
        columns=column_order,
        fill_value=0,
    )
)


# ------------------------------------------------------------
# 使用对数色阶
# ------------------------------------------------------------

# Manhattan内部订单量远高于其他组合。
# 如果直接使用原始订单量，其他格子的颜色会几乎一样。
# 因此颜色使用log10(订单量+1)，文字仍显示真实订单量。
borough_od_log = np.log10(
    borough_od_matrix + 1
)

# # 注释以万条为单位
# annotation_matrix = (
#     borough_od_matrix
#     .applymap(
#         lambda value: (
#             f"{value / 10000:.1f}万"
#             if value >= 10000
#             else f"{value:,.0f}"
#         )
#     )
# )

# 将热力图中的订单量转换为便于阅读的文本
# 大于等于1万条时以“万”为单位，其余显示原始数量
annotation_matrix = (
    borough_od_matrix
    .map(
        lambda value: (
            f"{value / 10000:.1f}万"
            if value >= 10000
            else f"{value:,.0f}"
        )
    )
)

fig, ax = plt.subplots(
    figsize=(9.5, 7)
)

sns.heatmap(
    borough_od_log,
    annot=annotation_matrix,
    fmt="",
    cmap="YlOrRd",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={
        "label": "订单量对数色阶",
        "shrink": 0.82,
    },
    ax=ax,
)

ax.set_title(
    "行政区之间的出租车订单流向",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel("下车行政区")
ax.set_ylabel("上车行政区")

ax.tick_params(
    axis="x",
    rotation=30,
)

ax.tick_params(
    axis="y",
    rotation=0,
)

add_source_note(
    fig,
    "单元格显示真实订单量，颜色采用对数色阶",
)

fig.tight_layout(
    rect=[0, 0.055, 1, 1]
)

heatmap_output_path = (
    FIGURE_DIR
    / "09_borough_od_heatmap.png"
)

fig.savefig(
    heatmap_output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ============================================================
# 九、检查图片保存情况
# ============================================================

output_files = [
    quadrant_output_path,
    flow_output_path,
    airport_output_path,
    heatmap_output_path,
]

print("\n" + "=" * 70)
print("高级经营分析图表生成完成")
print("=" * 70)

for file_path in output_files:
    status = (
        "保存成功"
        if file_path.exists()
        else "保存失败"
    )

    print(f"{file_path.name}：{status}")

print(f"\n图片保存目录：{FIGURE_DIR.resolve()}")